In [ ]:
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv 

load_dotenv()

model = init_chat_model(
    model_provider="google_genai", 
                        model ="gemini-2.5-flash",
                        temperature=0.7,
                        max_output_tokens=1024,
                        top_p=0.95,
                        top_k=40,
                        timeout = 30,
                        max_retries = 3,
                        load_env=True
)

In [ ]:
from dataclasses import dataclass
from langchain.agents import create_agent
from langchain.tools import tool, ToolRuntime
from langchain_core.utils.uuid import uuid7
from langgraph.checkpoint.memory import InMemorySaver

In [ ]:
USER_DATABASE = {
    "user123": {
        "name": "Alice Johnson",
        "account_type": "Premium",
        "balance": 5000,
        "email": "alice@example.com",
    },
    "user456": {
        "name": "Bob Smith",
        "account_type": "Standard",
        "balance": 1200,
        "email": "bob@example.com",
    },
}

In [ ]:
@dataclass 
class UserContext:
    user_id : str

In [ ]:
@tool
def get_account_info(runtime: ToolRuntime[UserContext]) -> str:
    """Get the current user's account information."""

    stream_writer = runtime.stream_writer

    stream_writer("Looking up your account...")

    user_id = runtime.context.user_id

    if user_id in USER_DATABASE:
        stream_writer("Account found. Retrieving account details...")

        user = USER_DATABASE[user_id]

        result = (
            f"Account holder: {user['name']}\n"
            f"Type: {user['account_type']}\n"
            f"Balance: ${user['balance']}"
        )

        stream_writer("Account information retrieved.")

        return result

    stream_writer("User account was not found.")
    return "User not found"

In [ ]:
agent = create_agent(
    model,
    tools = [get_account_info],
    context_schema = UserContext,
    checkpointer = InMemorySaver(),
    system_prompt = "You are a helpful financial assistant. You can provide account information based on the user context provided."
)

In [ ]:
thread_id = str(uuid7())
config = {"configurable": {"thread_id": thread_id}}
context = UserContext(user_id="user123")

In [ ]:
result = agent.invoke(
    {"messages": [{"role": "user", "content": "What's my current balance?"}]},
    config=config,
    context=context,
)

In [ ]:
print(result["messages"][-1].content_blocks)